# Two-stage detector: Faster R-CNN

COCO-pretrained Faster R-CNN ResNet50-FPN v2 (torchvision) on the adopted split: training setup, history, results, tuning experiments, what the region proposal network and the second stage contribute, and failure analysis. The structure is the template of notebooks 03 to 05.

| # | Section |
| --- | --- |
| 1 | [Setup](#1-setup) |
| 2 | [Training setup](#2-training-setup) |
| 3 | [Training history](#3-training-history) |
| 4 | [Results](#4-results) |
| 5 | [Experiments](#5-experiments) |
| 6 | [Inside the model](#6-inside-the-model) |
| 7 | [Failure analysis](#7-failure-analysis) |
| 8 | [Comparison](#8-comparison) |
| 9 | [Conclusions](#9-conclusions) |

## 1. Setup

Imports, data root from `DATA_ROOT`, the shared plot theme, the annotations of the adopted split, and the Faster R-CNN rows of `experiments/results.csv`, the source of truth for reported numbers. Run directories come from the `run_dir` column; section 6.3 also needs their `best.pt`, which exists only on the PC.

In [ ]:
import json
import sys
import tomllib
from pathlib import Path

import numpy as np
import polars as pl
import torch
from decouple import AutoConfig
from IPython.display import display
from torchvision.models.detection.faster_rcnn import FasterRCNN

# make the repository root importable when the kernel starts in notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
# decouple locates .env from the caller file, which is a temp file inside a kernel
config = AutoConfig(search_path=PROJECT_ROOT)

from src.data.annotations import AnnotationTables, load_annotations, load_split_clips
from src.data.boxes import explode_size_buckets
from src.data.classes import CLASS_NAMES
from src.data.dataset import select_frames
from src.data.statistics import compute_resize_scale
from src.evaluate import METRICS_FILE, PREDICTIONS_FILE
from src.evaluation.failure_analysis import (
    DetectionMatches,
    compute_proposal_recall,
    compute_recall_by_size,
    count_error_categories,
    find_best_f1_threshold,
    match_detections,
    select_error_examples,
)
from src.inference.torchvision_detectors import (
    MODEL_STATE_KEY,
    select_device,
    tabulate_rpn_proposals,
)
from src.models.anchors import compute_best_anchor_iou, tabulate_anchors
from src.models.faster_rcnn import assign_roi_levels, build_faster_rcnn
from src.training.experiment import CONFIG_FILE
from src.visualization.frames import find_densest_crop, plot_crop_panels, plot_error_examples
from src.visualization.plots import plot_grouped_bars, plot_training_history
from src.visualization.style import GT_COLOR, PREDICTION_COLOR, set_plot_theme
from src.visualization.tables import compare_with_reference, load_run_histories

SEED = 42
set_plot_theme()
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(200)

DATA_ROOT = Path(config("DATA_ROOT"))
FAMILY = "faster_rcnn"
BASELINE = "faster_rcnn-baseline"
SEED_REPEAT = "faster_rcnn-baseline_5ep-s7"
tables = load_annotations(DATA_ROOT, load_split_clips(PROJECT_ROOT / "config" / "data.toml"))
val_images = tables.images.filter(pl.col("split") == "val")
val_objects = tables.objects.filter(pl.col("split") == "val")
results = pl.read_csv(PROJECT_ROOT / "experiments" / "results.csv")
family_results = results.filter(pl.col("experiment").str.starts_with(f"{FAMILY}-"))
run_dirs = {
    row["experiment"]: PROJECT_ROOT / row["run_dir"] for row in family_results.iter_rows(named=True)
}
run_configs = {e: tomllib.loads((d / CONFIG_FILE).read_text()) for e, d in run_dirs.items()}
run_metrics = {e: json.loads((d / METRICS_FILE).read_text()) for e, d in run_dirs.items()}
family_results.select("experiment", "run_dir", "seed", "machine", "is_dirty")

Five Faster R-CNN runs, all on the PC from a clean working tree: the baseline, three one-factor experiments, and one seed repeat (seed 7).

## 2. Training setup

The baseline protocol shared with the other families (AGENTS.md, Data and Experiments): COCO-pretrained weights, frames resized to a shorter side of 1024 px (`max_size` 1824), train frames at step 5, about 1.5 GPU hours on the PC, and deliberately simple optimization: SGD with momentum 0.9 at a constant LR without warmup or augmentation. Values below are read from the baseline run's resolved `config.toml` and `metrics.json`.

In [ ]:
baseline_config = run_configs[BASELINE]
baseline_training = run_metrics[BASELINE]["training"]
train_frames = select_frames(
    tables.images.filter(pl.col("split") == "train"), baseline_training["frame_step"]
)
setup_fields = {
    "architecture": baseline_config["arch"],
    "initialization": f"{baseline_config['init']} (new 9-class box predictor)",
    "trainable backbone stages": baseline_config["trainable_backbone_layers"],
    "frozen BatchNorm": baseline_config["freeze_batchnorm"],
    "input size (min_size, max_size, px)": f"{baseline_config['min_size']}, "
    f"{baseline_config['max_size']}",
    "train frames (step)": f"{train_frames.height} (step {baseline_training['frame_step']})",
    "val frames for monitoring (step)": baseline_training["val_frame_step"],
    "epochs": baseline_training["epochs"],
    "batch size x gradient accumulation": f"{baseline_training['batch_size']} x "
    f"{baseline_training['grad_accum']}",
    "effective batch size": baseline_training["effective_batch_size"],
    "optimizer": f"SGD, momentum {baseline_config['momentum']}",
    "learning rate, schedule, warmup": f"{baseline_config['lr']}, {baseline_config['schedule']}, "
    f"{baseline_config['warmup_iters']} iterations",
    "weight decay": baseline_config["weight_decay"],
    "mixed precision": baseline_config["amp"],
    "seed": baseline_config["seed"],
    "machine": baseline_training["machine"],
    "GPU hours": round(baseline_training["gpu_hours"], 2),
    "best epoch (monitoring mAP)": baseline_training["best_epoch"],
}
pl.DataFrame({"setting": list(setup_fields), "value": [str(v) for v in setup_fields.values()]})

- 4,919 train frames (step 5) and 10 epochs at effective batch 4 took 1.33 GPU hours, within the 1.5 hour budget.
- The best monitoring epoch is 2 of 10.

In [ ]:
head_settings = [
    (table, key, str(value))
    for table in ("rpn", "roi_heads", "inference")
    for key, value in baseline_config[table].items()
]
pl.DataFrame(head_settings, schema=["table", "setting", "value"], orient="row")

All RPN and RoI head settings are torchvision defaults. Only the inference settings differ: score threshold 0.001 instead of 0.05 and 500 detections per frame instead of 100, so that AP integrates the precision-recall curve up to the leaderboard limit.

## 3. Training history

Per-epoch monitoring on val frames at step 5 (520 frames) with the shared evaluator; `best.pt` is the epoch with the highest monitoring mAP. All Faster R-CNN runs are shown; the 5-epoch runs use the baseline's LR, data order, and seed, so their first epochs are directly comparable with the baseline's.

In [ ]:
histories = load_run_histories(run_dirs)
monitoring_metrics = {
    "val/map": "Monitoring mAP@[.5:.95]",
    "val/map50": "Monitoring AP50",
    "val/ap_small": "Monitoring AP_small",
}
fig = plot_training_history(
    histories, monitoring_metrics, "Faster R-CNN monitoring metrics per epoch (val, step 5)"
)

- Every run peaks within its first 4 epochs: baseline 0.3024 at epoch 2, then 0.2757 to 0.2975 over epochs 3 to 10.
- `lr_cosine` peaks at epoch 2 as well (0.3067) and ends at 0.2769, below the baseline's 0.2837: decaying the LR to 0 does not stop the decline.
- Monitoring AP50 falls almost steadily after epoch 1 in both 10-epoch runs (baseline 0.6175 to 0.5689, `lr_cosine` 0.6113 to 0.5483).
- The seed repeat peaks at epoch 1 (0.3026) instead of 2: the best epoch itself varies with the seed.

In [ ]:
loss_metrics = {
    "train/loss_objectness": "RPN objectness loss",
    "train/loss_rpn_box_reg": "RPN box loss",
    "train/loss_classifier": "Second-stage classification loss",
    "train/loss_box_reg": "Second-stage box loss",
}
fig = plot_training_history(
    histories.filter(pl.col("experiment") != "faster_rcnn-giou_loss"),
    loss_metrics,
    "Faster R-CNN training losses per epoch (mean over the epoch's batches)",
)

- All four training losses fall in every epoch while the monitoring mAP falls after epoch 2. The model keeps fitting the 4,919 training frames from 56 clips, without augmentation, but stops improving on val; this is consistent with overfitting (hypothesis).
- `anchors_small` has lower RPN losses and higher second-stage losses. Its smaller anchors change which anchors are positive and which proposals reach the second stage, so its losses are not comparable with the other runs.
- `giou_loss` is left out because its second-stage box loss is a different function.

In [ ]:
histories.group_by("experiment", maintain_order=True).agg(
    pl.len().alias("epochs"),
    pl.col("epoch").get(pl.col("val/map").arg_max()).alias("best_epoch"),
    pl.col("val/map").max().round(4).alias("best_monitoring_map"),
    pl.col("val/map").last().round(4).alias("last_monitoring_map"),
    pl.col("epoch_seconds").mean().round(0).alias("seconds_per_epoch"),
    pl.col("gpu_memory_gb").max().round(2).alias("peak_gpu_memory_gb"),
)

Epoch time is 464 to 492 s and peak memory 8.5 GB in every run: the experiments change no cost of training.

## 4. Results

Rows of `experiments/results.csv`: the shared evaluator on all 2,593 val frames with `best.pt`, `maxDets` 500, ignored regions as `iscrowd=1`. `map_max_width_1920` and `ap_small_max_width_1920` use only val frames at most 1920 px wide, the test size range (secondary metrics). ms/image is measured on the PC at batch 1, including the resize and NMS.

In [ ]:
RESULT_COLUMNS = [
    "experiment", "epochs", "effective_batch_size", "gpu_hours", "map", "map50", "map75",
    "ap_small", "ap_medium", "ap_large", "map_max_width_1920", "ap_small_max_width_1920",
    "ms_per_image", "public_lb",
]
family_results.select(RESULT_COLUMNS).with_columns(pl.col("gpu_hours", "ms_per_image").round(2))

- All Faster R-CNN runs lie within 0.0053 mAP of the baseline (0.3013 to 0.3066) and 0.0156 AP_small (0.1069 to 0.1225).
- ms/image is 60.8 to 64.8 for the same architecture, so differences of a few ms are timing noise.
- The 5-epoch runs used half the GPU hours of the baseline (0.65 to 0.69) because the baseline's best epoch is 2.

## 5. Experiments

Each experiment changes one factor against the baseline. The note of each row states the change and the hypothesis. Seed noise comes from `faster_rcnn-baseline_5ep-s7`, the baseline's first 5 epochs with seed 7 instead of 42.

In [ ]:
family_results.select("experiment", "epochs", "notes")

In [ ]:
COMPARED_METRICS = ["map", "map50", "map75", "ap_small", "ap_medium", "ap_large"]
compare_with_reference(family_results, BASELINE, COMPARED_METRICS).with_columns(
    pl.selectors.starts_with("delta_").round(4)
)

- `giou_loss`: +0.0053 mAP, +0.0099 AP50, +0.0020 AP75, +0.0134 AP_small, +0.0180 AP_medium, −0.0104 AP_large.
- `lr_cosine`: +0.0049 mAP, −0.0077 AP50, +0.0092 AP75, +0.0156 AP_small.
- `anchors_small`: +0.0024 mAP, +0.0038 AP_small, −0.0142 AP_medium.
- The GIoU loss was meant to improve localization at high IoU, but its gain is at AP50 (+0.0099), not AP75 (+0.0020).
- Smaller anchors barely change AP_small (+0.0038) although they cover small boxes much better (section 6.1).

In [ ]:
COMMON_EPOCHS = 5
monitoring_by_epoch = (
    histories.filter(pl.col("epoch") <= COMMON_EPOCHS)
    .pivot(on="experiment", index="epoch", values="val/map")
    .sort("epoch")
)
monitoring_deltas = monitoring_by_epoch.select(
    "epoch",
    *(
        (pl.col(e) - pl.col(BASELINE)).round(4).alias(f"{e} - baseline")
        for e in run_dirs
        if e != BASELINE
    ),
)
seed_difference = pl.col(f"{SEED_REPEAT} - baseline").abs()
mean_seed_difference, max_seed_difference = monitoring_deltas.select(
    seed_difference.mean(), seed_difference.max().alias("max")
).row(0)
display(monitoring_deltas)
pl.DataFrame(
    {
        "seed noise measure": [
            "full val mAP, seed 7 - seed 42",
            "monitoring mAP, mean |seed 7 - seed 42| over epochs 1 to 5",
            "monitoring mAP, max |seed 7 - seed 42| over epochs 1 to 5",
        ],
        "value": [
            round(
                family_results.filter(pl.col("experiment") == SEED_REPEAT)["map"].item()
                - family_results.filter(pl.col("experiment") == BASELINE)["map"].item(),
                4,
            ),
            round(mean_seed_difference, 4),
            round(max_seed_difference, 4),
        ],
    }
)

- Seed noise: the full-val mAP of seed 7 differs from seed 42 by 0.0009, and their monitoring mAP by 0.0054 per epoch on average (at most 0.0119). AP_small differs by 0.0092 on full val.
- The experiments gain 0.0024 to 0.0053 full-val mAP: more than the full-val seed difference, less than the per-epoch spread. With one seed each, none of them is established (AGENTS.md: small gains count as noise until they repeat with another seed).
- The AP_small gains of `lr_cosine` (+0.0156) and `giou_loss` (+0.0134) are about 1.5 times the seed difference of AP_small.

In [ ]:
# the tuned model: highest full-val mAP among the experiments, seed repeats excluded
TUNED = str(
    family_results.filter(~pl.col("experiment").str.contains(r"-s\d+$"))
    .sort("map", descending=True)["experiment"]
    .first()
)
TUNED

The tuned model is `faster_rcnn-giou_loss` (full-val mAP 0.3066), 0.0004 above `lr_cosine` (0.3062); the order of the two is within noise. It is the reference for notebooks 03 to 05.

## 6. Inside the model

Faster R-CNN has two stages. The region proposal network (RPN) scores every anchor on the FPN levels P2 to P6 for objectness and regresses it; after NMS at IoU 0.7 it keeps 1,000 proposals per frame at inference. The second stage pools a 7 × 7 feature map per proposal from one of P2 to P5 (RoI Align), classifies it into 8 classes or background, and regresses a class-specific box; per-class NMS at IoU 0.5 and the 500-detection limit follow. This section looks at what each stage sees: anchor coverage of the box shapes (6.1), the pooling level of each object (6.2), and how many objects the proposals already cover before the second stage (6.3, 6.4).

### 6.1 Anchor coverage

Best IoU of every `train` box with any anchor of four settings, with anchors at their feature-map grid centers (the IoU that the RPN matcher sees; `src.models.anchors.compute_best_anchor_iou`). Boxes are resized like the model input (`min_size` 1024, `max_size` 1824); size buckets use the original box area. The RPN calls anchors with IoU ≥ 0.7 positives and < 0.3 negatives, but also keeps the best anchor of every GT box as a positive (`allow_low_quality_matches`), so a box below 0.7 still trains one anchor; below 0.3 that anchor starts far from the box.

- `default`: torchvision, sizes 32 to 512, h / w 0.5, 1, 2 (3 per location), the baseline.
- `anchors_small`: sizes 16 to 256, 3 per location (experiment `faster_rcnn-anchors_small`).
- `two_sizes`: sizes (16, 24) to (256, 384), 6 per location (not run: the RPN head changes shape and loses its COCO weights).
- `anchors_small_ratio_1.5`: `anchors_small` with h / w 1.5 added, 4 per location (not run).

In [ ]:
FASTER_RCNN_STRIDES = [4, 8, 16, 32, 64]
RATIOS = [0.5, 1.0, 2.0]
anchor_settings = {
    "default": (baseline_config["rpn"]["anchor_sizes"], baseline_config["rpn"]["aspect_ratios"]),
    "anchors_small": (
        run_configs["faster_rcnn-anchors_small"]["rpn"]["anchor_sizes"],
        run_configs["faster_rcnn-anchors_small"]["rpn"]["aspect_ratios"],
    ),
    "two_sizes": ([[16, 24], [32, 48], [64, 96], [128, 192], [256, 384]], [RATIOS] * 5),
    "anchors_small_ratio_1.5": ([[16], [32], [64], [128], [256]], [[0.5, 1.0, 1.5, 2.0]] * 5),
}
anchor_tables = {
    name: tabulate_anchors(sizes, ratios, FASTER_RCNN_STRIDES)
    for name, (sizes, ratios) in anchor_settings.items()
}
train_boxes = (
    tables.objects.filter(pl.col("split") == "train")
    .join(tables.images.select("split", "image_id", "width", "height"), on=["split", "image_id"])
    .with_columns(
        compute_resize_scale(baseline_config["min_size"], baseline_config["max_size"]).alias(
            "scale"
        )
    )
    .with_columns(pl.col("x", "y", "w", "h") * pl.col("scale"))
)
train_box_array = train_boxes.select("x", "y", "w", "h").to_numpy()
best_anchor_iou = train_boxes.select("area").with_columns(
    pl.Series(name, compute_best_anchor_iou(train_box_array, anchors, is_grid_aligned=True))
    for name, anchors in anchor_tables.items()
)
IOU_THRESHOLDS = (0.7, 0.5, 0.3)
anchor_coverage = (
    pl.concat(
        [
            best_anchor_iou.with_columns(pl.lit("all").alias("size_bucket")),
            explode_size_buckets(best_anchor_iou, pl.col("area")).with_columns(
                pl.col("size_bucket").cast(pl.String)
            ),
        ]
    )
    .drop("area")
    .unpivot(index="size_bucket", variable_name="anchor_setting", value_name="best_iou")
    .group_by("anchor_setting", "size_bucket", maintain_order=True)
    .agg(
        pl.len().alias("boxes"),
        pl.col("best_iou").mean().round(3).alias("mean_best_iou"),
        *((pl.col("best_iou") < t).mean().round(3).alias(f"share_below_{t}") for t in IOU_THRESHOLDS),
    )
    .sort(
        pl.col("anchor_setting").cast(pl.Enum(list(anchor_settings))),
        pl.col("size_bucket").cast(pl.Enum(["all", "small", "medium", "large"])),
    )
)
anchor_coverage

- Default anchors: 10.5% of train boxes have best grid IoU below 0.5, 37.0% of small boxes; 56.8% are below 0.7.
- `anchors_small`: below 0.5 falls to 2.0% (small 3.6%), but below 0.7 rises to 68.0%: halving every size leaves medium boxes between two sizes that are an octave apart.
- `two_sizes` (6 per location): 0.8% below 0.5 (small 2.8%) and 28.5% below 0.7, the best coverage, at the cost of a new RPN head.
- Adding h / w 1.5 to `anchors_small` lowers the share below 0.5 by 0.4 points (2.0% to 1.6%; small 3.6% to 3.4%).
- These numbers reproduce the data-analysis estimates in `docs/experiment_ideas.md`. Since `anchors_small` gained only +0.0038 AP_small with much better coverage, anchor coverage is not the main limit for small objects (hypothesis; see sections 6.2 and 7.1).

### 6.2 Feature level of each object

The box head pools every proposal from one FPN level, floor(4 + log2(√area / 224)) clamped to P2 to P5 (FPN equation 1, torchvision `LevelMapper`; `src.models.faster_rcnn.assign_roi_levels`). Shown for `val` GT boxes after the resize, as a proxy for the proposals that match them.

In [ ]:
val_boxes = (
    val_objects.join(val_images.select("image_id", "width", "height"), on="image_id")
    .with_columns(
        compute_resize_scale(baseline_config["min_size"], baseline_config["max_size"]).alias(
            "scale"
        )
    )
    .with_columns(pl.col("x", "y", "w", "h") * pl.col("scale"))
)
roi_levels = val_boxes.with_columns(
    pl.Series("level", assign_roi_levels(val_boxes.select("x", "y", "w", "h").to_numpy()))
)
level_shares = (
    pl.concat(
        [
            roi_levels.with_columns(pl.lit("all").alias("size_bucket")),
            explode_size_buckets(roi_levels, pl.col("area")).with_columns(
                pl.col("size_bucket").cast(pl.String)
            ),
        ]
    )
    .with_columns(pl.format("P{}", "level").alias("level"))
    .group_by("size_bucket", "level")
    .len()
    .with_columns((pl.col("len") / pl.col("len").sum().over("size_bucket")).round(3).alias("share"))
    .pivot(on="level", index="size_bucket", values="share", sort_columns=True)
    .fill_null(0.0)
    .sort(pl.col("size_bucket").cast(pl.Enum(["all", "small", "medium", "large"])))
)
level_shares

- 96.3% of val GT boxes map to P2 (stride 4): all small and 99.4% of medium boxes. Large boxes map to P2 (71.4%), P3 (25.3%), and P4 (3.3%); none map to P5.
- On COCO the box head sees objects spread over P2 to P5; here it sees almost all objects through P2 at the 7 × 7 RoI Align resolution, so a 20 px object covers about 5 × 5 cells of P2.

### 6.3 Proposal recall

Proposals of the RPN on every 25th `val` frame (about 4% of the frames, all 7 clips), for the baseline, the RPN experiment `anchors_small`, and the tuned model. A GT box counts as covered by the top n proposals if one of them has IoU ≥ 0.5 (or 0.7) with it, regardless of class. The last columns give the recall of the final detections on the same frames (class-aware, IoU 0.5, all kept detections), so the gap between the two is what the second stage loses or adds. This cell loads `best.pt` and needs the PC.

In [ ]:
PROPOSAL_FRAME_STEP = 25
PROPOSAL_EXPERIMENTS = list(dict.fromkeys([BASELINE, "faster_rcnn-anchors_small", TUNED]))
TOP_NS = [100, 300, 1000]
proposal_frames = select_frames(val_images, PROPOSAL_FRAME_STEP)
device = select_device()


def load_detector(experiment: str) -> FasterRCNN:
    detector = build_faster_rcnn(run_configs[experiment])
    checkpoint = torch.load(run_dirs[experiment] / "best.pt", map_location="cpu", weights_only=True)
    detector.load_state_dict(checkpoint[MODEL_STATE_KEY])
    return detector.to(device).eval()


proposals = {
    experiment: tabulate_rpn_proposals(load_detector(experiment), proposal_frames, device)
    for experiment in PROPOSAL_EXPERIMENTS
}
pl.DataFrame(
    {
        "experiment": PROPOSAL_EXPERIMENTS,
        "frames": [proposal_frames.height] * len(PROPOSAL_EXPERIMENTS),
        "proposals_per_frame": [
            round(proposals[e].height / proposal_frames.height, 1) for e in PROPOSAL_EXPERIMENTS
        ],
    }
)

In [ ]:
val_predictions = {e: pl.read_parquet(run_dirs[e] / PREDICTIONS_FILE) for e in PROPOSAL_EXPERIMENTS}
matches = {e: match_detections(tables, "val", val_predictions[e]) for e in PROPOSAL_EXPERIMENTS}
proposal_frame_ids = proposal_frames["image_id"].implode()
proposal_objects = val_objects.filter(pl.col("image_id").is_in(proposal_frame_ids))
recall_parts = []
for experiment in PROPOSAL_EXPERIMENTS:
    rpn_recall = compute_proposal_recall(proposal_objects, proposals[experiment], TOP_NS, [0.5, 0.7])
    frame_matches = DetectionMatches(
        matches[experiment].detections,
        matches[experiment].objects.filter(pl.col("image_id").is_in(proposal_frame_ids)),
    )
    recall_parts += [
        rpn_recall.select(
            pl.lit(experiment).alias("experiment"),
            "size_bucket",
            "objects",
            pl.format("rpn_top{}_iou{}", "top_n", "iou_threshold").alias("measure"),
            "recall",
        ),
        compute_recall_by_size(frame_matches).select(
            pl.lit(experiment).alias("experiment"),
            "size_bucket",
            "objects",
            pl.lit("detections_iou0.5").alias("measure"),
            "recall",
        ),
    ]
recall_table = (
    pl.concat(recall_parts, how="vertical_relaxed")
    .with_columns(pl.col("recall").round(3))
    .pivot(on="measure", index=["experiment", "size_bucket", "objects"], values="recall")
)
recall_table

*To be filled after the run on the PC: this output needs the `best.pt` checkpoints.*

### 6.4 Proposals on a dense frame

The proposal frame with the most GT boxes, cropped to the 480 × 270 px window with the most small objects. Left: GT boxes; middle: the top 300 proposals of the tuned model; right: its detections at the best-F1 score threshold of section 7.2.

In [ ]:
tuned_threshold = find_best_f1_threshold(matches[TUNED])["threshold"]
dense_frame = (
    proposal_objects.group_by("image_id")
    .len()
    .join(proposal_frames, on="image_id")
    .sort("len", "image_id", descending=[True, False])
    .row(0, named=True)
)
is_dense_frame = pl.col("image_id") == dense_frame["image_id"]
dense_objects = val_objects.filter(is_dense_frame)
CROP_SIZE = (480, 270)
crop_corner = find_densest_crop(
    dense_objects, (dense_frame["width"], dense_frame["height"]), CROP_SIZE, 32**2
)
fig = plot_crop_panels(
    Path(dense_frame["file_path"]),
    tables.ignored.filter(pl.col("split") == "val", is_dense_frame),
    crop_corner,
    CROP_SIZE,
    [
        (f"Ground truth, {dense_objects.height} boxes in the frame", dense_objects, GT_COLOR),
        ("Top 300 proposals", proposals[TUNED].filter(is_dense_frame, pl.col("rank") < 300),
         PREDICTION_COLOR),
        (f"Detections with score ≥ {tuned_threshold:.2f}",
         val_predictions[TUNED].filter(is_dense_frame, pl.col("score") >= tuned_threshold),
         PREDICTION_COLOR),
    ],
    f"{TUNED} on {dense_frame['clip']} frame {dense_frame['frame_id']}",
)

*To be filled after the run on the PC: this output needs the `best.pt` checkpoints.*

## 7. Failure analysis

Baseline and tuned model on all `val` frames, from their `val_predictions.parquet` (`src.evaluation.failure_analysis`). Detections are matched greedily in descending score order, class-aware, at IoU 0.5; detections at least half inside an ignored region are excluded from the counts.

### 7.1 Recall by object size

Share of GT boxes matched by any kept detection (up to 500 per frame, score ≥ 0.001), per COCO size bucket. This is the recall ceiling of the AP computation at IoU 0.5.

In [ ]:
ANALYSED = list(dict.fromkeys([BASELINE, TUNED]))
recall_by_size = pl.concat(
    compute_recall_by_size(matches[e]).with_columns(pl.lit(e).alias("experiment"))
    for e in ANALYSED
)
fig = plot_grouped_bars(
    recall_by_size,
    "size_bucket",
    "recall",
    "experiment",
    ("COCO size bucket", "Recall at IoU 0.5", "Recall of all kept detections by object size (val)"),
    "{:.3f}",
)
recall_by_size.with_columns(pl.col("recall").round(3))

- With all kept detections the tuned model matches 82.7% of val GT boxes at IoU 0.5: 66.4% of small, 84.8% of medium, and 97.0% of large boxes (baseline 81.6%: 65.9%, 83.6%, 96.7%).
- One small object in three is not detected at any score, which caps AP_small.

### 7.2 Error breakdown at the best-F1 threshold

The score threshold that maximizes F1 at IoU 0.5 on `val` (ignored detections excluded), then one category per detection above it: correct, duplicate (same-class GT already matched), wrong class, poor localisation (best IoU in [0.1, 0.5)), background (best IoU < 0.1); misses are GT boxes left unmatched.

In [ ]:
thresholds = pl.DataFrame(
    [{"experiment": e, **find_best_f1_threshold(matches[e])} for e in ANALYSED]
).with_columns(pl.col("threshold", "precision", "recall", "f1").round(3))
error_counts = pl.concat(
    count_error_categories(matches[e], thresholds.filter(pl.col("experiment") == e)["threshold"].item())
    .with_columns(pl.lit(e).alias("experiment"))
    for e in ANALYSED
)
display(thresholds)
error_counts.pivot(on="experiment", index="category", values="count")

In [ ]:
fig = plot_grouped_bars(
    error_counts,
    "category",
    "count",
    "experiment",
    ("Category", "Detections or missed GT boxes", "Error breakdown at the best-F1 threshold (val)"),
    "{:.0f}",
)

- Best-F1 thresholds: 0.392 for the tuned model (precision 0.728, recall 0.658, F1 0.691) and 0.396 for the baseline (0.719, 0.633, 0.673).
- At these thresholds misses dominate: 35,976 of 105,240 GT boxes (34.2%) for the tuned model. False positives split almost evenly into background (9,375), poor localisation (8,312), and wrong class (7,904); duplicates are negligible (240), so per-class NMS at IoU 0.5 is not a problem.
- Against the baseline, `giou_loss` has 2,617 fewer misses and 1,072 fewer poorly localised detections, but 1,134 more background detections.

In [ ]:
tuned_threshold = thresholds.filter(pl.col("experiment") == TUNED)["threshold"].item()
CLASS_BY_ID = dict(enumerate(CLASS_NAMES))
(
    matches[TUNED].detections.filter(
        pl.col("score") >= tuned_threshold, pl.col("category") == "wrong_class"
    )
    .group_by("best_class_id", "class_id")
    .len()
    .sort("len", "best_class_id", "class_id", descending=[True, False, False])
    .head(8)
    .select(
        pl.col("best_class_id").replace_strict(CLASS_BY_ID).alias("gt_class"),
        pl.col("class_id").replace_strict(CLASS_BY_ID).alias("predicted_class"),
        pl.col("len").alias("detections"),
    )
)

- The largest confusions are van predicted as car (1,987) and motor predicted as person (1,867), then bicycle and motor confused both ways (889 and 558).
- Van and car are visually similar from above. A motor GT box and its rider's person box overlap, so a person detection can reach IoU 0.5 with the motor box only.

### 7.3 Example frames per category

Tuned model at its best-F1 threshold: the four highest-scoring detections of every error category and four missed GT boxes drawn at random (seed 42), at most one per clip. Each crop is centered on the example, with all GT boxes (yellow), all kept detections (blue), and shaded ignored regions; the example has a thick outline.

In [ ]:
examples = select_error_examples(
    matches[TUNED], val_images, tuned_threshold, per_category=4, seed=SEED
)
fig = plot_error_examples(
    examples,
    val_images,
    val_objects,
    tables.ignored.filter(pl.col("split") == "val"),
    val_predictions[TUNED].filter(pl.col("score") >= tuned_threshold),
    context_scale=5.0,
    min_side=96,
)
examples.join(
    val_images.select("image_id", "clip", "frame_id"), on="image_id", maintain_order="left"
).select(
    "category",
    "clip",
    "frame_id",
    pl.col("class_id").replace_strict(CLASS_BY_ID).alias("class_name"),
    pl.col("score").round(3),
    (pl.col("w") * pl.col("h")).sqrt().round(1).alias("sqrt_area_px"),
)

- Correct: large, unoccluded vehicles score close to 1.
- Duplicate: groups of people (basketball court, pedestrians) and one car with two boxes.
- Wrong class: all four are vans predicted as car, with scores above 0.98.
- Poor localisation: IoU 0.44 to 0.50 with a GT box of the same class, just below the threshold. In three crops the detection fits the car tightly while the GT box is visibly larger than the car.
- Background: confident car and truck detections on vehicles that have neither a GT box nor an ignored region (the white car at the intersection, the car at night).
- Miss: a car hidden in vegetation, a person standing next to another person, a motor under its rider, and a motor in clutter next to an ignored region.

### 7.4 Failure cases

Four failure cases of the tuned model at its best-F1 threshold. Measured counts come from the tables above; the causes are hypotheses drawn from the crops in section 7.3.

1. **Small objects are missed** (35,976 misses; recall of small boxes 66.4% with all detections, against 97.0% for large ones). Almost every object is pooled from P2 (section 6.2), so a 20 px object gives the box head only a few feature cells. Smaller anchors did not change this (+0.0038 AP_small with `anchors_small`), which points to feature resolution rather than anchor coverage. Higher input resolution or SAHI tiling are candidates for the final pipeline.
2. **Confusion between similar classes**: van as car (1,987 detections) and the two-wheeler classes (bicycle and motor, 1,447 both ways). The classes differ in size and shape details that are small from a drone's view. A rider's person box overlaps the motor box, which gives motor as person (1,867). Class-aware evaluation counts each as both a false positive and a miss.
3. **Loose or inconsistent GT boxes** produce confident poor localisation errors (8,312, including detections scored 0.99). The four example detections reach IoU 0.44 to 0.50 with a same-class GT box, and in the crops the detection is tight around the car while the GT box includes margin, so the object is found but counted as a false positive and a miss.
4. **Unlabelled vehicles** produce confident background detections (9,375). Several crops show real cars detected with score 0.98 or higher, with no GT box and no ignored region. They cost precision on val, and on the test set as well if it is labelled the same way. A higher score threshold would not remove them, since their scores are as high as those of correct detections.

## 8. Comparison

Baseline against the tuned model by class and by size. Later notebooks compare their families against the tuned Faster R-CNN; the last table lists the current baseline rows of all families for reference.

In [ ]:
per_class_ap = pl.DataFrame(
    {
        "class_name": list(CLASS_NAMES),
        **{
            e: [round(run_metrics[e]["evaluation"]["per_class_ap"][name], 4) for name in CLASS_NAMES]
            for e in ANALYSED
        },
    }
)
per_class_ap.with_columns((pl.col(TUNED) - pl.col(BASELINE)).round(4).alias("delta"))

- `giou_loss` gains most on van (+0.0445), tricycle (+0.0257), motor (+0.0146), and bicycle (+0.0131), and loses on bus (−0.0429) and truck (−0.0147).
- Bus has only 2 val clips and truck 15 tracks (`01_eda.ipynb` sections 9.2 and 11.3), so their per-class changes are noisy.

In [ ]:
compare_with_reference(
    family_results.filter(pl.col("experiment").is_in(ANALYSED)),
    BASELINE,
    ["map", "map50", "map75", "ap_small", "ap_medium", "ap_large", "ms_per_image"],
).with_columns(pl.selectors.float().round(4))

The tuned model gains mostly on small and medium boxes (+0.0134 AP_small, +0.0180 AP_medium) and loses on large boxes (−0.0104). Inference cost is unchanged: the GIoU loss changes training only.

In [ ]:
results.filter(pl.col("experiment").str.ends_with("-baseline")).select(
    "experiment", "epochs", "gpu_hours", "map", "map50", "ap_small", "ap_medium", "ap_large",
    "ms_per_image", "public_lb",
).with_columns(pl.col("gpu_hours", "ms_per_image").round(2))

- Among the baselines, Faster R-CNN (0.3013 mAP) is second to RT-DETRv2 (0.3209) and has the highest AP50 (0.6159).
- It is also the slowest at inference: 62.5 ms/image, against 53.2 for RT-DETRv2, 48.6 for RetinaNet, and 13.7 for YOLO11s.

## 9. Conclusions

To be written after the run on the PC.